# Sampling Techniques & Measures of Central Tendency

## Topic Overview & Learning Objectives

This notebook serves as an exhaustive, sub-atomic reference and interview-readiness guide for statistical sampling methodologies and measures of central tendency. It covers theoretical mechanics, mathematical foundations, edge cases, error profiling, algorithmic complexity, and production-grade Python implementations using `pandas` and `numpy`.

### Learning Objectives:
1. **Probability Sampling Frameworks**: Master Simple Random Sampling, Stratified Sampling, Cluster Sampling, and Systematic Sampling.
2. **Taxonomy of Biases & Errors**: Distinguish Selection Bias, Non-Response Bias, Sampling Error, and Non-Sampling Error.
3. **Measures of Central Tendency**: Deep dive into Arithmetic Mean, Median, and Mode, evaluating distributional skewness, mathematical sensitivity, and outlier robustness.
4. **Computational & Algorithmic Profiling**: Understand vectorization, time/space complexity ($O(N)$, $O(N \log N)$), memory overhead, and implementation tradeoffs in large-scale data engineering.
5. **Applied Production Simulations**: Compare sample estimators against true population parameters across structured multi-store retail and manufacturing datasets.


## Section 2: Exhaustive Concept Inventory


### Core Sampling Foundations
* Population vs. Sample & Inferential Estimation
* Simple Random Sampling (SRS)
* Stratified Sampling Mechanics
* Cluster Sampling Mechanics
* Systematic Sampling & The Sampling Interval $k$
* Comparative Mechanics: Stratified Sampling vs. Cluster Sampling

### Pandas & NumPy Implementation Methods
* `pandas.DataFrame.sample()` Mechanics (`n`, `frac`, `replace`, `weights`, `random_state`)
* `pandas.core.groupby.GroupBy.apply()` in Stratified Slicing
* `group_keys=False` Operational Behavior in Pandas
* `numpy.random.choice()` Execution (`size`, `replace`, probability distributions)
* `pandas.Series.unique()` vs. `nunique()` in Cluster Identification
* `pandas.Series.isin()` Filtering Mechanics
* `pandas.DataFrame.iloc` Strided Slicing (`[::k, :]`)

### Biases, Discrepancies & Estimator Errors
* Selection Bias Mechanics
* Non-Response Bias & Attrition
* Sampling Error vs. Non-Sampling Error
* The Law of Large Numbers (LLN) & Central Limit Theorem (CLT)
* Sample Estimator Bias: Actual Population Mean vs. Sample Mean (Unbiasedness)

### Measures of Central Tendency
* Arithmetic Mean: Mathematical Mechanics & Linear Sensitivity
* `pandas.Series.mean()` vs. `numpy.mean()` (Handling of `NaN` / Missing Values)
* Median: Order Statistics & Sorting Mechanics
* `pandas.Series.median()` vs. `numpy.median()` (Handling Even vs. Odd $N$ and `NaN`)
* Mode: Frequency Distribution Mechanics & Multi-modality
* `pandas.Series.mode()` Zero-Indexing (`.mode()[0]`) & Empty Set Mechanics
* Impact of Distributional Skewness (Right-Skewed, Left-Skewed, Symmetric)

### Computational Complexity & Practical Exercises
* Time & Space Complexity of Central Tendency & Sampling Algorithms
* Beginner & Intermediate Applied Coding Challenges
* Senior/Lead Technical Interview Architectural Coding Challenge

## Sub-Atomic Concept Q&A with Validated Code Examples

---

### Concept 1: Population vs. Sample & Inferential Estimation

**Question:**
What is the theoretical distinction between a population parameter and a sample statistic, and why is inferential sampling mathematically necessary in big data engineering?

**Answer:**
- **Theoretical Intuition:** A *population* represents the totality of all possible observations, events, or entities conforming to a defined specification (parameter denoted by Greek letters, e.g., $\mu$ for mean, $\sigma$ for standard deviation). A *sample* is a subset drawn from the population (statistic denoted by Latin letters, e.g., $\bar{x}$, $s$).
- **Technical Mechanics:** Inferential statistics uses sample statistics as point estimators or interval estimators to make probabilistic statements about true population parameters without exhausting computational, network, or storage resources.
- **Complexity:** Computing exact population parameters on petabyte-scale distributed datasets (e.g., full scans on Delta Lake/Snowflake) requires $O(N)$ distributed compute. Rigorous sampling reduces this to $O(n)$ where $n \ll N$.
- **Practical Relevance:** Production pipeline optimization, A/B test evaluation, data validation checks, and low-latency feature engineering.

Sigma | Effect
--- | ---
0.2 |Less variation
0.5 | Moderate variation
0.75 | More variation
1.5 | Much greater variation

Interview tip: A log-normal distribution is obtained when the logarithm of a random variable follows a normal distribution.

Remember: 
* Normal distribution = symmetrical values. Log-normal distribution = positive values with a right-skewed shape.
* Without replacement → sample size cannot exceed population size.
  * replace=False: Once an item is selected, it cannot be selected again (no duplicates).
  * replace=True: The same item can be selected multiple times (duplicates allowed).


In [1]:
import pandas as pd
import numpy as np

# Set deterministic seed
np.random.seed(42)

# Simulate Population: 1,000,000 transactions
N = 1_000_000 # Python ignores underscores in numeric literals. They improve readability.
# lognormal generates positive skewed data, which is common for income distributions.
population_income = np.random.lognormal( 
    mean=10.5, # Mean of underlying normal distribution (mean of the logarithms of the generated values.)
    sigma=0.75, # Standard deviation of underlying normal distribution
    size=N # Number of random values to generate
) 
# Population parameter: True Population Mean (mu)
true_mu = np.mean(population_income)

# Sample statistic: Sample Mean (x_bar) with n=1,000
n = 1_000
sample_income = np.random.choice(population_income, size=n, replace=False)
sample_x_bar = np.mean(sample_income)

print(f"True Population Mean (mu):     ${true_mu:,.2f}")
print(f"Sample Statistic Mean (x_bar): ${sample_x_bar:,.2f}")
print(f"Absolute Estimation Error:     ${abs(true_mu - sample_x_bar):,.2f}")


True Population Mean (mu):     $48,048.68
Sample Statistic Mean (x_bar): $46,797.40
Absolute Estimation Error:     $1,251.28


---

### Concept 2: Simple Random Sampling (SRS)

**Question:**
What constitutes a Simple Random Sample (SRS), and what is the difference between Simple Random Sampling With Replacement (SRSWR) and Without Replacement (SRSWOR)?

**Answer:**
- **Theoretical Intuition:** In SRS, every individual item in the population of size $N$ has an identical inclusion probability of $p = 1/N$. Furthermore, every combination of $n$ distinct elements has an equal probability of being chosen: $P(S) = \frac{1}{\binom{N}{n}}$.
- **Technical Mechanics:**
  - **SRSWOR (Without Replacement):** Once an element is sampled, it cannot be selected again. Draws are dependent; probabilities change at each step: $P(E_k) = \frac{1}{N - k + 1}$.
  - **SRSWR (With Replacement):** Elements are returned after selection. Draws are independent; inclusion probability remains $1/N$ at every step.
- **Return Behavior:** An unordered subset dataframe of shape `(n, num_columns)`.
- **Computational Profiling:** Reservoir sampling algorithm runs in $O(N)$ time and $O(n)$ auxiliary space.


In [2]:
# Setup standard reference population dataframe
np.random.seed(0)
df_population = pd.DataFrame({
    'CustomerID': range(1, 1001),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], 1000),
    'Satisfaction': np.random.randint(1, 6, 1000),
    'Income': np.random.randint(20000, 80000, 1000)
})

# SRS Without Replacement (default)
srs_wor = df_population.sample(n=100, replace=False, random_state=42)

# SRS With Replacement
srs_wr = df_population.sample(n=100, replace=True, random_state=42)

print("SRSWOR Unique IDs:", srs_wor['CustomerID'].nunique())
print("SRSWR Unique IDs (Expect <= 100 due to collisions):", srs_wr['CustomerID'].nunique())


SRSWOR Unique IDs: 100
SRSWR Unique IDs (Expect <= 100 due to collisions): 94


---

### Concept 3: Stratified Sampling Mechanics

**Question:**
How does Stratified Sampling work, what constitutes a 'stratum', and why does it systematically produce estimators with lower variance than Simple Random Sampling for heterogeneous populations?

**Answer:**
- **Theoretical Intuition:** The population is partitioned into mutually exclusive, collectively exhaustive subsets called *strata* ($S_1, S_2, \dots, S_L$) based on a known auxiliary variable (e.g., Region, Tier, Income Bracket). Independent random samples are then drawn within each stratum.
- **Technical Mechanics:** Strata are constructed to be *internally homogeneous* (low within-stratum variance $\sigma_h^2$) and *mutually heterogeneous* (high between-stratum variance). Because the sampling variance of stratified sampling is $\text{Var}(\bar{y}_{st}) = \sum \left(\frac{N_h}{N}\right)^2 \frac{s_h^2}{n_h}$, eliminating between-stratum variance directly minimizes overall standard error.
- **Proportional vs. Disproportional:**
  - *Proportional Allocation*: $n_h = n \cdot (N_h / N)$.
  - *Neyman/Optimal Allocation*: Allocates larger samples to strata with larger variance and lower sampling costs.


In [3]:

# 1. Fixed Allocation (up to 25 records per Region)
stratified_sample_fixed = pd.concat([
    group.sample(n=min(len(group), 25), random_state=1)
    for _, group in df_population.groupby('Region', as_index=False)
], ignore_index=True)

# 2. Proportional Allocation (10% from each Region)
stratified_sample_prop = (
    df_population.groupby('Region', group_keys=False)
    .sample(frac=0.10, random_state=1)
    .reset_index(drop=True)
)

print("Fixed Allocation Sample per Region:\n", stratified_sample_fixed['Region'].value_counts())
print("\nProportional Allocation Sample per Region:\n", stratified_sample_prop['Region'].value_counts())

Fixed Allocation Sample per Region:
 Region
East     25
North    25
South    25
West     25
Name: count, dtype: int64

Proportional Allocation Sample per Region:
 Region
North    26
South    25
West     25
East     24
Name: count, dtype: int64


---

### Concept 4: Cluster Sampling Mechanics

**Question:**
Explain the architecture of Cluster Sampling. How do the internal and external characteristics of clusters differ fundamentally from strata?

**Answer:**
- **Theoretical Intuition:** The population is divided into natural heterogeneous groupings called *clusters* (e.g., classrooms, geographic zip codes, server nodes, manufacturing batches). A random sample of entire clusters is selected, and *every* unit within the selected clusters is measured (Single-Stage Cluster Sampling).
- **Technical Mechanics:**
  - Clusters must be *internally heterogeneous* (each cluster ideally mirrors the mini-diversity of the entire population) and *externally homogeneous* (clusters are interchangeable in overall statistical characteristics).
  - High operational and logistical efficiency when constructing a full population sampling frame is impossible or cost-prohibitive.
- **Drawback:** Higher sampling error (design effect $> 1$) compared to SRS if intra-cluster correlation (ICC / $\rho$) is high.


In [4]:
# Simulate Manufacturing Batch Dataset
np.random.seed(42)
num_records = 5000
df_manufacturing = pd.DataFrame({
    'PartID': range(1001, 1001 + num_records),
    'BatchNumber': np.random.choice([f'Batch_{i:02d}' for i in range(1, 21)], size=num_records),
    'ToleranceError_um': np.random.normal(loc=0.0, scale=1.5, size=num_records)
})

# Cluster Sampling: Randomly select 5 batches and sample ALL parts in those batches
all_batches = df_manufacturing['BatchNumber'].unique()
selected_batches = np.random.choice(all_batches, size=5, replace=False)
cluster_sample = df_manufacturing[df_manufacturing['BatchNumber'].isin(selected_batches)]

print("Selected Cluster Batches:", selected_batches)
print("Total rows in cluster sample:", len(cluster_sample))
print("Sample distribution across selected batches:\n", cluster_sample['BatchNumber'].value_counts())


Selected Cluster Batches: ['Batch_03' 'Batch_02' 'Batch_07' 'Batch_08' 'Batch_09']
Total rows in cluster sample: 1249
Sample distribution across selected batches:
 BatchNumber
Batch_08    274
Batch_03    272
Batch_02    252
Batch_09    235
Batch_07    216
Name: count, dtype: int64


---

### Concept 5: Systematic Sampling & The Sampling Interval $k$

**Question:**
How is Systematic Sampling formulated, how is the interval $k$ calculated, and what catastrophic trap (periodicity) must engineers guard against?

**Answer:**
- **Theoretical Intuition:** Elements are selected at a constant periodic interval from an ordered list. Starting from a randomly selected initial index $r \in [0, k-1]$, every $k$-th element is systematically extracted: $r, r+k, r+2k, \dots$
- **Technical Mechanics:** The sampling interval is $k = \lfloor N / n \rfloor$.
- **Periodicity Vulnerability (Critical Interview Trap):** If the underlying data contains periodic hidden cycles, seasonal rhythms, or repetitive patterns matching interval $k$ (e.g., machines recalibrating every 10th cycle, sales spikes every 7th day), systematic sampling captures an intensely biased, non-representative sample.
- **Complexity:** Selection logic runs in $O(n)$ time via direct memory striding.


In [5]:
# Systematic Sampling with random start index r
N = len(df_population)
n = 100
k = N // n  # Sampling interval = 10

# Generate random start
random_start = np.random.randint(0, k)
systematic_sample_random_start = df_population.iloc[random_start::k, :]

print(f"Population size N={N}, Desired sample n={n}, Interval k={k}")
print(f"Random start index: {random_start}")
print(f"Sampled size: {len(systematic_sample_random_start)}")
print(f"Sampled Customer IDs (first 5): {systematic_sample_random_start['CustomerID'].head().tolist()}")


Population size N=1000, Desired sample n=100, Interval k=10
Random start index: 5
Sampled size: 100
Sampled Customer IDs (first 5): [6, 16, 26, 36, 46]


---

### Concept 6: Comparative Mechanics: Stratified vs. Cluster Sampling

**Question:** In a technical architecture review, provide a comparative breakdown contrasting Stratified Sampling and Cluster Sampling across selection units, internal/external variance, variance efficiency, and use cases.

**Answer:**
The two methods serve complementary objectives: Stratified Sampling improves representation and statistical precision, whereas Cluster Sampling primarily improves cost and logistical efficiency.

| Feature / Dimension      | Stratified Sampling                                                                                      | Cluster Sampling                                                                                                                           |
| ------------------------ | -------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------ |
| Primary Selection Unit   | Individual members are selected from every stratum.                                                      | Entire clusters are selected from the population.                                                                                          |
| Subgroup Inclusion       | All strata are included; samples are drawn from each stratum.                                            | Only selected clusters are included. In one-stage cluster sampling, all members of selected clusters are surveyed.                         |
| Within-Group Homogeneity | Members within each stratum should be relatively homogeneous regarding the characteristic being studied. | Members within each cluster should ideally be heterogeneous, representing a mini-population.                                               |
| Between-Group Variance   | Ideally, high between-stratum variance, so strata are meaningfully different from one another.           | Ideally, low between-cluster variance, so clusters resemble one another.                                                                   |
| Statistical Efficiency   | Generally higher precision and lower sampling variance than SRS when strata are well designed.           | Generally lower precision than stratified sampling at the same sample size, especially when members within clusters are highly correlated. |
| Cost & Logistics         | Requires information to classify population members into strata; may involve higher preparation costs.   | Often lower cost because data collection is concentrated in selected locations or groups.                                                  |
| Primary Risk             | Incorrect stratification or poor allocation of samples across strata.                                    | High intra-cluster correlation (ICC), which reduces the effective sample size.                                                             |

### Important Technical Corrections

1\. Variance assumptions

- Stratified Sampling: Low within-stratum variance and high between-stratum variance are desirable.
- Cluster Sampling: High within-cluster diversity and low between-cluster variance are desirable.

2\. Statistical efficiency

- Stratified sampling generally reduces sampling variance because it ensures representation from every stratum.
- Cluster sampling can increase sampling variance when individuals within the same cluster are similar (high ICC).

3\. Types of Cluster Sampling

- One-stage Cluster Sampling: Select clusters and survey every member within them.
- Two-stage Cluster Sampling: Select clusters, then randomly select individual members within those clusters.

Therefore, saying all members are selected applies only to one-stage cluster sampling.

### Practical Use Cases

Stratified Sampling:

A university wants representation from every department. It divides students into departments and randomly selects students from each department.

Cluster Sampling:

A research team wants to survey households across a large region. It randomly selects villages and surveys households in the selected villages.

### Interview Takeaway

- Stratified Sampling: Select some individuals from every group.
- Cluster Sampling: Select some groups, then survey individuals within selected groups.

Remember: Stratified sampling focuses primarily on improving statistical precision, while cluster sampling focuses primarily on reducing data collection costs.

In [6]:
# Comparative Demonstration of Group Selection
# Stratified: Samples from EVERY group
stratified_groups = stratified_sample_fixed['Region'].unique()
# Cluster: Samples ALL records from SOME groups
cluster_groups = cluster_sample['BatchNumber'].unique()

print(f"Total Regions in Population: {df_population['Region'].nunique()}")
print(f"Regions represented in Stratified Sample: {len(stratified_groups)} (100% of strata)")

print(f"\nTotal Batches in Manufacturing Population: {df_manufacturing['BatchNumber'].nunique()}")
print(f"Batches represented in Cluster Sample: {len(cluster_groups)} (Only 5 selected clusters)")


Total Regions in Population: 4
Regions represented in Stratified Sample: 4 (100% of strata)

Total Batches in Manufacturing Population: 20
Batches represented in Cluster Sample: 5 (Only 5 selected clusters)


---

### Concept 7: `pandas.DataFrame.sample()` Mechanics

**Question:**
Examine the signature of `pandas.DataFrame.sample()`. What are the behavioral roles of `n`, `frac`, `replace`, `weights`, and `random_state`?

**Answer:**
- **Signature:** `df.sample(n=None, frac=None, replace=False, weights=None, random_state=None, axis=None, ignore_index=False)`
- **`n` vs. `frac`:**
  - `n` (int): Absolute count of items to return. Mutually exclusive with `frac`.
  - `frac` (float): Proportion of axis items to return ($0.0 \le \text{frac} \le 1.0$, or $> 1.0$ if `replace=True`). Raises `ValueError` if both `n` and `frac` are set.
- **`replace` (bool):** Sample with or without replacement. If `False`, specifying $n > N$ raises `ValueError: Cannot take a larger sample than population when 'replace=False'`.
- **`weights` (str or array-like):** Assigns non-uniform sampling probabilities. Must sum to $>0$ and have non-negative values. Automatically normalized.
- **`random_state` (int, RandomState, or None):** Seeds the PRNG for reproducibility across runs.


In [7]:
# Validating parameter permutations and error handling
sample_n = df_population.sample(n=5, random_state=42)
sample_frac = df_population.sample(frac=0.005, random_state=42)

# Weighted sampling: Bias towards higher satisfaction scores
weights = df_population['Satisfaction'] / df_population['Satisfaction'].sum()
weighted_sample = df_population.sample(n=100, weights=weights, random_state=42)

print("Sample with n=5 rows:", len(sample_n))
print("Sample with frac=0.005 rows:", len(sample_frac))
print("Mean satisfaction in regular sample:", df_population.sample(n=100, random_state=42)['Satisfaction'].mean())
print("Mean satisfaction in weighted sample:", weighted_sample['Satisfaction'].mean())

# Error demonstrating n > N without replacement
try:
    df_population.head(10).sample(n=20, replace=False)
except ValueError as e:
    print("\nExpected Error Caught:", e)


Sample with n=5 rows: 5
Sample with frac=0.005 rows: 5
Mean satisfaction in regular sample: 2.92
Mean satisfaction in weighted sample: 3.67

Expected Error Caught: Cannot take a larger sample than population when 'replace=False'


---

### Concept 8: `GroupBy.apply()` in Stratified Slicing

**Question:**
How does `pandas.DataFrame.groupby().apply()` perform stratified sampling internally, why is `min(len(x), n)` necessary when sampling without replacement, and what roles do `include_groups=False` and `.reset_index(level=0)` play in the pipeline?

**Answer:**

* **Internal Execution (Split-Apply-Combine):** Calling `.groupby('StrataColumn')` partitions the DataFrame into group-specific subsets mapped by unique category keys. `.apply(callable)` iterates through each subset `x`, executes the sampling function independently, and concatenates the resulting DataFrames back together along the index via `pd.concat()`.
* **Preventing Sparse-Stratum Crashes (`min(len(x), n)`):** By default, `.sample()` samples without replacement (`replace=False`). If a rare stratum contains fewer rows than the requested sample size $n$ (for example, 5 rows when $n=10$), pandas raises `ValueError: Cannot take a larger sample than population`. Clamping the sample size with `min(len(x), n)` safely caps the draw at the available group size instead of breaking the pipeline.
* **Avoiding Deprecation Warnings (`include_groups=False`):** Added in pandas 2.2, `include_groups=False` excludes the grouping column (`Category`) from the subset `x` passed into the `lambda` function. Because `groupby().apply()` already tracks the group keys and automatically attaches them to the outer level of the output's MultiIndex during concatenation, passing the grouping column inside `x` is redundant and triggers a `DeprecationWarning` in modern pandas versions.
* **Restoring the Column Structure (`.reset_index(level=0)`):** Concatenating grouped results produces a two-level **MultiIndex**: `(Category, original_row_index)`. Calling `.reset_index(level=0)` promotes only the outer index level (`Category`) back into a regular DataFrame column while preserving each sampled row's original index in `level=1` for traceability. *(Note: using `.reset_index(drop=True)` instead would discard the original row indices altogether).*



In [8]:
# Sparse Strata Simulation
df_imbalanced = pd.DataFrame({
    'Category': ['A']*100 + ['B']*50 + ['C']*5,
    'Value': np.random.randn(155)
})

# Robust stratified sampling using min(len(x), k)
target_per_group = 10
stratified_robust = (
    df_imbalanced.groupby('Category')
    .apply(lambda x: x.sample(n=min(len(x), target_per_group), random_state=42), include_groups=False)
    .reset_index(level=0)
)

print("Imbalanced Strata Sample Counts:\n", stratified_robust['Category'].value_counts())


Imbalanced Strata Sample Counts:
 Category
A    10
B    10
C     5
Name: count, dtype: int64


This alternative approach—shuffling the entire DataFrame first with `.sample(frac=1)` and then taking the top rows per group with `.groupby().head()`—is widely considered the **idiomatic, high-performance pandas pattern** for stratified sampling.

Compared to the `groupby().apply()` approach, it offers five distinct advantages:

### 1. Massive Speed & Vectorization Gains

* **Previous (`apply`):** Executes a Python `lambda` loop once per group, constructs intermediate sub-DataFrames for every stratum, and runs `pd.concat()` at the end. On datasets with thousands of groups (e.g., user IDs or ZIP codes), this Python-level overhead creates a major bottleneck.
* **New (`sample` + `head`):** Uses purely **Cython-optimized, vectorized operations**. Shuffling happens in a single pass, and `.head(n)` simply slices the first $n$ integer positions per group using pre-computed group indexer arrays. It routinely runs **10x to 100x faster** on high-cardinality strata.

### 2. Built-in Edge-Case Safety (No `min(len(x), n)` Needed)

* **Previous:** `.sample(n)` throws a `ValueError` if a group has fewer rows than $n$, forcing you to write defensive `min(len(x), target_per_group)` logic.
* **New:** `.head(n)` naturally returns *up to* $n$ rows. If stratum `'C'` only has 5 rows when `target_per_group = 10`, `.head(10)` returns all 5 rows without raising an error or requiring extra boilerplate.

### 3. Cleaner Index & Schema Preservation

* **Previous:** `groupby().apply()` generates a two-level MultiIndex `(Category, original_index)`, requiring `include_groups=False` (to dodge pandas 2.2+ deprecation warnings) and `.reset_index(level=0)` (to restore `Category` as a column, which also reorders `Category` to the first column position).
* **New:** Passing `as_index=False` (or simply omitting it, since `.head()` preserves the original index and columns by default) keeps the exact original column order, data types, and original row indices intact without any index manipulation or version-specific flags.

### 4. Global Randomization vs. Per-Group Seed Reuse

* **Previous:** Passing `random_state=42` inside the `lambda` resets the random number generator to the exact same seed for every group, which can introduce subtle positional correlation if strata are structured identically.
* **New:** `df.sample(frac=1, random_state=42)` shuffles the entire dataset once using a single, continuous random stream before grouping, guaranteeing true uniform randomness across the whole table.

---

### Side-by-Side Comparison

| Feature | `.groupby().apply(lambda x: x.sample(...))` | `.sample(frac=1).groupby().head(n)` |
| --- | --- | --- |
| **Execution Engine** | Python-level loop + `pd.concat()` | Vectorized Cython slice |
| **Small Stratum Behavior** | Crashes unless wrapped in `min(len(x), n)` | Safely returns all available rows |
| **Index / Column Handling** | Creates MultiIndex; needs `reset_index(level=0)` | Preserves original columns and index as-is |
| **Pandas 2.2+ Compatibility** | Requires `include_groups=False` | No deprecation warnings |
| **Best Use Case** | Weighted sampling or sampling *with replacement* | Standard uniform stratified sampling |

*(Note: The only scenario where the `apply()` pattern is still required is when you need **sampling with replacement** (`replace=True`), **row-level probability weights** (`weights=...`), or **proportional sampling** (`frac=0.2` per group rather than a fixed count `n`).)*

In [9]:
stratified_robust = (
    df_imbalanced.sample(frac=1, random_state=42)
    .groupby('Category', as_index=False)
    .head(target_per_group)
)

print("Imbalanced Strata Sample Counts (Alternative Method):\n", stratified_robust['Category'].value_counts())  

Imbalanced Strata Sample Counts (Alternative Method):
 Category
A    10
B    10
C     5
Name: count, dtype: int64


---

### Concept 9: `group_keys=False` Operational Behavior

**Question:**
Why is `group_keys=False` critical when executing `apply()` on a grouped object in modern Pandas versions?

**Answer:**
- **Technical Mechanics:** By default (or in historical Pandas versions), `groupby().apply()` prepends the grouping key to the resulting DataFrame's index, creating a `MultiIndex` `(group_key, original_index)`.
- **Behavioral Flag:** Setting `group_keys=False` instructs Pandas **not** to add the grouping keys to the index, returning a single index that preserves original row indices. This prevents unnecessary downstream index flattening steps (`.reset_index(level=0, drop=True)`).


In [10]:
# Comparing index structure with group_keys True vs False
# res_with_keys = df_population.groupby('Region', group_keys=True).apply(lambda x: x.sample(2, random_state=1), include_groups=False).reset_index(level=0)
res_with_keys = df_population.groupby('Region', group_keys=True).apply(lambda x: x.sample(2, random_state=1))
res_without_keys = df_population.groupby('Region', group_keys=False).apply(lambda x: x.sample(2, random_state=1))
print("Index with group_keys=True:\n", res_with_keys.index)
print("\nIndex with group_keys=False:\n", res_without_keys.index)


Index with group_keys=True:
 MultiIndex([( 'East', 296),
            ( 'East', 997),
            ('North', 693),
            ('North', 125),
            ('South', 300),
            ('South', 998),
            ( 'West', 245),
            ( 'West', 988)],
           names=['Region', None])

Index with group_keys=False:
 Index([296, 997, 693, 125, 300, 998, 245, 988], dtype='int64')


---

### Concept 10: `numpy.random.choice()` Execution

**Question:**
What are the mechanics, parameter defaults, and performance considerations of `numpy.random.choice()`?

**Answer:**
- **Signature:** `numpy.random.choice(a, size=None, replace=True, p=None)`
- **Parameters:**
  - `a`: 1-D array-like or int. If an int, the random sample is generated over `np.arange(a)`.
  - `size`: Output shape (int or tuple of ints).
  - `replace`: Boolean. **Default is `True`** (unlike `pandas.sample()` where default is `False`!). This is a major interview gotcha.
  - `p`: The probabilities associated with each entry in `a`. If not given, uniform distribution is assumed.
- **Complexity:** $O(k)$ for uniform sampling without replacement using Knuth-style algorithms; weighted sampling without replacement requires $O(n + k \log n)$.


In [11]:
clusters = np.array(['Cluster_A', 'Cluster_B', 'Cluster_C', 'Cluster_D'])

# Default behavior: replace=True (Notice default difference from Pandas!)
draw_with_replacement = np.random.choice(clusters, size=3)

# Distinct selection: replace=False
draw_without_replacement = np.random.choice(clusters, size=3, replace=False)

print("Default (replace=True):", draw_with_replacement)
print("Explicit replace=False:", draw_without_replacement)


Default (replace=True): ['Cluster_D' 'Cluster_D' 'Cluster_B']
Explicit replace=False: ['Cluster_D' 'Cluster_B' 'Cluster_C']


---

### Concept 11: `pandas.Series.unique()` vs. `nunique()`

**Question:**
What are the internal execution and return type differences between `Series.unique()` and `Series.nunique()`?

**Answer:**
- **`Series.unique()`:** Returns a NumPy ndarray or ExtensionArray of unique values in order of appearance. Time complexity: $O(N)$ amortized using hash table. **Includes `NaN` values** as a distinct element.
- **`Series.nunique(dropna=True)`:** Returns an integer count of unique elements. By default, `dropna=True`, meaning `NaN` values are **excluded** from the count.
- **Gotcha:** If a column has `[1, 2, np.nan]`, `len(s.unique()) == 3`, but `s.nunique() == 2`.


In [12]:
s = pd.Series(['North', 'South', 'North', np.nan, 'East'])

print("s.unique():\n", s.unique(), "\nType:", type(s.unique()))
print("\ns.nunique() [dropna=True]:", s.nunique())
print("s.nunique(dropna=False):", s.nunique(dropna=False))


s.unique():
 <ArrowStringArray>
['North', 'South', nan, 'East']
Length: 4, dtype: str 
Type: <class 'pandas.arrays.ArrowStringArray'>

s.nunique() [dropna=True]: 3
s.nunique(dropna=False): 4


---

### Concept 12: `pandas.Series.isin()` Filtering Mechanics

**Question:**
How does `Series.isin()` optimize filtering operations against sampled clusters, and what are its performance characteristics compared to iterative filtering?

**Answer:**
- **Technical Mechanics:** Evaluates whether each element in the Series is contained in the passed collection of values. Returns a boolean Series of identical length.
- **Performance:** Internally converts the passed sequence into a C-level hash set (`std::unordered_set`). Membership checks for each of the $N$ Series elements take $O(1)$ amortized time, yielding an overall vector runtime of $O(N + M)$ where $M$ is the size of the cluster list.
- **Contrast:** Iterative row filtering using `.apply(lambda row: row['x'] in clusters)` runs in Python bytecode, incurring massive interpreter overhead.


In [13]:
# Vectorized membership test
selected_clusters = ['North', 'East']
mask = df_population['Region'].isin(selected_clusters)
filtered_df = df_population[mask]

print(f"Total rows matched: {len(filtered_df)}")
print(f"Unique regions in result: {filtered_df['Region'].unique().tolist()}")


Total rows matched: 496
Unique regions in result: ['North', 'East']


---

### Concept 13: `pandas.DataFrame.iloc` Strided Slicing

**Question:**
How does Python slice syntax operate inside `DataFrame.iloc[start:stop:step, :]`, and what are the zero-copy implications?

**Answer:**
- **Syntax:** `df.iloc[start:stop:step, :]`. When selecting every 10th row: `df.iloc[::10, :]`.
  - `start`: 0-indexed position (default `0`).
  - `stop`: terminating index (default end of frame).
  - `step`: interval $k$.
- **Memory Mechanics:** In underlying NumPy/Pandas memory layout, strided slicing along axis 0 creates a view with an adjusted stride whenever memory contiguousness allows, avoiding immediate data buffer duplication until modifications or copies are triggered.
- **Complexity:** Slice creation is $O(1)$ metadata adjustment; subsequent iteration or copying is $O(N/k)$.


In [14]:
# Strided systematic slice
systematic_step_10 = df_population.iloc[::10, :]
print("Population Shape:", df_population.shape)
print("Systematic Sample Shape:", systematic_step_10.shape)
print("Indices chosen (first 5):", systematic_step_10.index[:5].tolist())


Population Shape: (1000, 4)
Systematic Sample Shape: (100, 4)
Indices chosen (first 5): [0, 10, 20, 30, 40]


---

### Concept 14: Selection Bias Mechanics

**Question:**
Define Selection Bias in statistical data collection. Provide concrete industry examples and explain its mathematical consequence on estimator validity.

**Answer:**
- **Theoretical Intuition:** Selection Bias occurs when the mechanism used to include units in a sample systematically favors or excludes certain segments of the population. Consequently, the probability of selection is correlated with the outcome variable $Y$.
- **Mathematical Impact:**
  $$\mathbb{E}[\bar{Y}_{sample}] \ne \mu$$
  The sample estimator is asymptotically biased. Increasing sample size $n \to \infty$ does **not** eliminate selection bias; it simply converges with high precision to the wrong value!
- **Industry Scenario:** Evaluating credit risk or income distribution by surveying individuals exclusively in premium shopping malls or wealthy residential districts.


In [15]:
# Demonstrating Selection Bias mathematically
# Population true income
true_mean = df_population['Income'].mean()

# Biased selection rule: Only high satisfaction customers respond/are sampled
biased_sample = df_population[df_population['Satisfaction'] >= 4]
biased_mean = biased_sample['Income'].mean()

print(f"Actual Population Mean Income:  ${true_mean:,.2f}")
print(f"Biased Selection Sample Mean:    ${biased_mean:,.2f}")
print(f"Systematic Bias Offset:          ${biased_mean - true_mean:,.2f}")


Actual Population Mean Income:  $49,529.36
Biased Selection Sample Mean:    $48,979.72
Systematic Bias Offset:          $-549.64


---

### Concept 15: Non-Response Bias & Attrition

**Question:**
What constitutes Non-Response Bias, how does it differ from voluntary response bias, and what methods are used to mitigate it?

**Answer:**
- **Theoretical Intuition:** Non-Response Bias occurs when the individuals who fail or refuse to participate in a survey/study systematically differ in relevant attributes from those who do participate.
- **Distinction:** Voluntary response bias occurs when participants self-select into a study (often those with extreme opinions); non-response bias occurs within a designated sample when non-respondents have a distinct distribution.
- **Mitigation Techniques:**
  1. Follow-up contact waves with non-respondents.
  2. Post-stratification weighting (raking).
  3. Inverse Probability Weighting (IPW) using propensity scoring models.


In [16]:
# Simulating Non-Response Bias: Lower income individuals drop out at 80% rate
np.random.seed(42)
df_survey = df_population.copy()
response_prob = np.where(df_survey['Income'] < 40000, 0.20, 0.85)
responded_mask = np.random.binomial(1, response_prob).astype(bool) # bino

df_responded = df_survey[responded_mask]

print(f"Full Survey Population Mean:   ${df_survey['Income'].mean():,.2f}")
print(f"Observed Respondent Mean:      ${df_responded['Income'].mean():,.2f}")
print(f"Response Rate:                 {len(df_responded)/len(df_survey):.1%}")


Full Survey Population Mean:   $49,529.36
Observed Respondent Mean:      $56,199.11
Response Rate:                 62.1%


---

### Concept 16: Sampling Error vs. Non-Sampling Error

**Question:**
In an enterprise statistical audit, how do you delineate Sampling Error from Non-Sampling Error?

**Answer:**
- **Sampling Error:**
  - The unavoidable variability that arises solely because a sample is observed rather than the entire population.
  - Under probability sampling, it is quantifiable via Standard Error: $SE = \frac{\sigma}{\sqrt{n}}$.
  - **Behavior:** Shrinks toward zero as $n \to N$ (obeying the square root law $\propto 1/\sqrt{n}$).
- **Non-Sampling Error:**
  - All errors arising from data collection, processing, and system defects: measurement errors, instrument miscalibration, data entry typos, leading questions, dishonest respondents, and data pipeline corruption.
  - **Behavior:** Present in both samples and full population censuses; does **not** diminish as $n$ increases, and frequently compounds with scale.


In [17]:
# Visualizing Sampling Error decay as n increases
sample_sizes = [10, 50, 100, 500, 1000]
simulated_errors = []

for s in sample_sizes:
    estimates = [df_population['Income'].sample(n=s).mean() for _ in range(50)]
    std_err = np.std(estimates)
    simulated_errors.append((s, std_err))

for s, err in simulated_errors:
    print(f"Sample Size n={s:4d} | Empirical Standard Error: ${err:,.2f}")


Sample Size n=  10 | Empirical Standard Error: $5,458.16
Sample Size n=  50 | Empirical Standard Error: $2,580.05
Sample Size n= 100 | Empirical Standard Error: $1,576.33
Sample Size n= 500 | Empirical Standard Error: $577.32
Sample Size n=1000 | Empirical Standard Error: $0.00


---

### Concept 17: Law of Large Numbers (LLN) & Central Limit Theorem (CLT)

**Question:**
State the operational mechanics of the Law of Large Numbers (LLN) and the Central Limit Theorem (CLT) as they apply to sample estimators.

**Answer:**
- **Law of Large Numbers (LLN):** States that as the sample size $n$ approaches infinity, the sample mean $\bar{X}_n$ converges almost surely (Strong Law) and in probability (Weak Law) to the true population mean $\mu$:
  $$\lim_{n \to \infty} P(|\bar{X}_n - \mu| < \epsilon) = 1$$
- **Central Limit Theorem (CLT):** Given independent and identically distributed (i.i.d.) random variables with mean $\mu$ and finite variance $\sigma^2$, the standardized distribution of the sample mean converges in distribution to a standard normal distribution, regardless of the underlying population distribution:
  $$\frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \xrightarrow{d} \mathcal{N}(0, 1)$$

Feature | LLN | CLT
 --- | --- | ---
Full form | Law of Large Numbers | Central Limit Theorem
Main idea | Average gets closer to population mean | Sample means form a normal distribution
Focus | Accuracy of average | Distribution of sample averages
What happens? | More observations improve average reliability | Repeated sample averages form a bell curve
Application | Estimating average sales | Statistical inference and confidence intervals


In [18]:
# Demonstrating Central Limit Theorem with highly skewed distribution
np.random.seed(42)
exponential_population = np.random.exponential(scale=50, size=500_000)

sample_means = [np.mean(np.random.choice(exponential_population, size=30, replace=False)) for _ in range(1000)]

print(f"Population Mean:         {np.mean(exponential_population):.4f}")
print(f"Mean of 1000 Samples:   {np.mean(sample_means):.4f}")
print(f"Theoretical SE (sigma/sqrt(n)): {np.std(exponential_population)/np.sqrt(30):.4f}")
print(f"Observed Sample Mean SD:        {np.std(sample_means):.4f}")


Population Mean:         50.0078
Mean of 1000 Samples:   49.7296
Theoretical SE (sigma/sqrt(n)): 9.1158
Observed Sample Mean SD:        8.8240


---

### Concept 18: Unbiasedness: Actual Population Mean vs. Sample Mean

**Question:**
Mathematically define an unbiased estimator. Is the sample mean $\bar{X}$ strictly unbiased for population mean $\mu$?

**Answer:**
- **Mathematical Definition:** An estimator $\hat{\theta}$ is unbiased for parameter $\theta$ if its expected value equals the parameter:
  $$\text{Bias}(\hat{\theta}) = \mathbb{E}[\hat{\theta}] - \theta = 0$$
- **Proof of Unbiasedness for Sample Mean:**
  $$\mathbb{E}[\bar{X}] = \mathbb{E}\left[\frac{1}{n} \sum_{i=1}^n X_i\right] = \frac{1}{n} \sum_{i=1}^n \mathbb{E}[X_i] = \frac{1}{n} (n \mu) = \mu$$
  Hence, under Simple Random Sampling, $\bar{X}$ is unconditionally an unbiased estimator of $\mu$.

Conclusion: The sample mean is unbiased because its expected value equals the population mean.

Note: Unbiased does not mean every sample gives the exact population mean. It means the estimator is correct on average over repeated sampling.

In [19]:
# Empirical validation of E[x_bar] == mu across 500 independent samples
num_simulations = 500
sample_size = 50
means = [df_population['Income'].sample(n=sample_size).mean() for _ in range(num_simulations)]

expected_sample_mean = np.mean(means)
actual_population_mean = df_population['Income'].mean()

print(f"True Population Mean (mu):           ${actual_population_mean:,.2f}")
print(f"Expected Value of Estimates E[x_bar]: ${expected_sample_mean:,.2f}")
print(f"Empirical Bias:                      ${expected_sample_mean - actual_population_mean:,.4f}")


True Population Mean (mu):           $49,529.36
Expected Value of Estimates E[x_bar]: $49,544.75
Empirical Bias:                      $15.3978


---

### Concept 19: Arithmetic Mean: Mathematical Mechanics & Linear Sensitivity

**Question:**
How is the Arithmetic Mean computed, what is its mathematical vulnerability, and why does a single extreme outlier corrupt the metric?

**Answer:**
- **Mathematical Formula:**
  $$\bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$$
- **Linear Sensitivity & Zero Breakdown Point:** The arithmetic mean has a breakdown point of $1/n \to 0$. Changing a single observation $x_k \to \infty$ forces $\bar{x} \to \infty$.
- **Minimization Property:** The arithmetic mean is the unique value $c$ that minimizes the sum of squared deviations: $\arg\min_c \sum_{i=1}^n (x_i - c)^2$.
- **Computational Profile:** Single pass $O(N)$ time complexity, $O(1)$ auxiliary space.


In [20]:
# Demonstrating breakdown point of arithmetic mean
clean_salaries = pd.Series([50000, 55000, 60000, 62000, 58000])
corrupted_salaries = clean_salaries.copy()
corrupted_salaries[4] = 50_000_000  # Extreme outlier injected

print(f"Original Mean:  ${clean_salaries.mean():,.2f}")
print(f"Corrupted Mean: ${corrupted_salaries.mean():,.2f}")


Original Mean:  $57,000.00
Corrupted Mean: $10,045,400.00


---

### Concept 20: `pandas.Series.mean()` vs. `numpy.mean()` (`NaN` Behavior)

**Question:**
What is the critical behavioral difference between `pandas.Series.mean()` and `numpy.mean()` when encountering missing (`NaN`) values?

**Answer:**
- **`pandas.Series.mean()`:** Defaults to `skipna=True`. It automatically excludes missing values from both numerator and denominator, computing the mean over non-null elements ($O(N)$).
- **`numpy.mean()`:** Defaults to treating `NaN` as infectious floating-point arithmetic. If an array contains even a single `np.nan`, `np.mean(arr)` returns `nan` unless `np.nanmean(arr)` is explicitly invoked.
- **Gotcha:** Passing a raw NumPy array containing `NaN` into downstream scoring pipelines will quietly propagate `NaN` throughout feature matrices.


In [21]:
data_with_nan = pd.Series([10.0, 20.0, np.nan, 30.0])
np_data = data_with_nan.to_numpy()

print("pandas .mean() default:", data_with_nan.mean())
print("numpy .mean() default: ", np.mean(np_data))
print("numpy .nanmean():      ", np.nanmean(np_data))


pandas .mean() default: 20.0
numpy .mean() default:  nan
numpy .nanmean():       20.0


---

### Concept 21: Median: Order Statistics & Sorting Mechanics

**Question:**
What is the Median, how is it derived mathematically for even vs. odd sample sizes, and what is its robustness profile?

**Answer:**
- **Mathematical Definition:** Given sorted order statistics $x_{(1)} \le x_{(2)} \le \dots \le x_{(n)}$:
  $$
  \text{Median} = \begin{cases}
  x_{\left(\frac{n+1}{2}\right)} & \text{if } n \text{ is odd} \\[6pt]
  \frac{x_{\left(\frac{n}{2}\right)} + x_{\left(\frac{n}{2} + 1\right)}}{2} & \text{if } n \text{ is even}
  \end{cases}
  $$
- **Minimization Property:** The median minimizes the sum of absolute deviations: $\arg\min_c \sum_{i=1}^n |x_i - c|$.
- **Robustness:** High breakdown point of $50\%$ (can tolerate up to half the sample being replaced with arbitrary noise without exploding).
- **Computational Profile:** Requires sorting $O(N \log N)$, or selection algorithm (Quickselect / Introselect) in $O(N)$ average time.


In [22]:
odd_series = pd.Series([12, 3, 5, 7, 100])      # n=5 -> sorted: [3, 5, 7, 12, 100] -> Median = 7
even_series = pd.Series([12, 3, 5, 7, 100, 10])   # n=6 -> sorted: [3, 5, 7, 10, 12, 100] -> Median = (7+10)/2 = 8.5

print("Odd n median: ", odd_series.median())
print("Even n median:", even_series.median())

# Demonstrating outlier resistance
print("Clean Salaries Median:    ", clean_salaries.median())
print("Corrupted Salaries Median:", corrupted_salaries.median())


Odd n median:  7.0
Even n median: 8.5
Clean Salaries Median:     58000.0
Corrupted Salaries Median: 60000.0


---

### Concept 22: `pandas.Series.median()` vs. `numpy.median()`

**Question:**
How do Pandas and NumPy handle missing values and linear interpolation during median calculations?

**Answer:**
- **`pandas.Series.median()`:** Automatically ignores `NaN` values by default (`skipna=True`). Uses linear interpolation when $n$ is even.
- **`numpy.median()`:** Propagates `NaN` unless `np.nanmedian()` is used.
- **Interpolation Parameters:** `np.percentile()` and modern NumPy versions allow configuring interpolation methods: `'linear'`, `'lower'`, `'higher'`, `'midpoint'`, `'nearest'`.


In [23]:
arr_with_nan = np.array([1, 2, np.nan, 4, 5])

print("Pandas median on Series:", pd.Series(arr_with_nan).median())
print("NumPy default median:   ", np.median(arr_with_nan))
print("NumPy nanmedian:        ", np.nanmedian(arr_with_nan))

data = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
print("\n50th Percentile (linear):", np.percentile(data, 50, method='linear'))   # 55.0
print("25th Percentile (lower):", np.percentile(data, 25, method='nearest'))    # 30


Pandas median on Series: 3.0
NumPy default median:    nan
NumPy nanmedian:         3.0

50th Percentile (linear): 55.0
25th Percentile (lower): 30


---

### Concept 23: Mode: Frequency Distribution Mechanics & Multi-modality

**Question:**
Define the statistical Mode. How is multi-modality handled, and for what data types is the mode the only permissible measure of central tendency?

**Answer:**
- **Theoretical Intuition:** The Mode is the data value that appears with highest frequency in a collection.
- **Data Types:** For **nominal categorical data** (e.g., Blood Type, Department, Region, HTTP Status Code), Mean and Median are mathematically undefined; Mode is the **only** legitimate measure of central tendency.
- **Modality Spectrum:**
  - *Unimodal*: A single dominant peak.
  - *Bimodal / Multimodal*: Two or more distinct peaks with equal highest frequency.
  - *No Mode*: Uniform distributions where all unique values occur with equal count.


In [24]:
# Categorical nominal data mode
car_colors = pd.Series(['Red', 'Blue', 'Black', 'Blue', 'White', 'Blue', 'Black'])
print("Dominant car color mode:", car_colors.mode(),sep="\n")

# Bimodal numerical series
bimodal_series = pd.Series([10, 20, 20, 30, 40, 40, 50])
print("\nBimodal result (multiple modes):", bimodal_series.mode(),sep="\n")


Dominant car color mode:
0    Blue
dtype: str

Bimodal result (multiple modes):
0    20
1    40
dtype: int64


---

### Concept 24: `pandas.Series.mode()` Zero-Indexing (`.mode()[0]`) & Empty Set Mechanics

**Question:**
Why does `pandas.Series.mode()` return a `Series` rather than a scalar, why do engineers write `.mode()[0]`, and what fatal bug can this trigger?

**Answer:**
- **Return Type:** Because a distribution can be multimodal, `Series.mode()` must return a `Series` containing all values having maximum frequency, sorted ascending.
- **The `.mode()[0]` Idiom:** Engineers extract the first modal value using `.mode()[0]`.
- **The Empty Set Crash (Interview Trap):** If the Series is empty or contains exclusively `NaN` values:
  `pd.Series([]).mode()` returns `Series([], dtype: float64)`.
  Accessing `[0]` immediately raises an unhandled `IndexError: single positional indexer is out-of-bounds`!
- **Defensive Production Pattern:**
  `mode_series = s.mode()`
  `val = mode_series.iloc[0] if not mode_series.empty else np.nan`


In [25]:
# Defensive Mode Extraction Pattern
empty_series = pd.Series([], dtype=float)

def get_safe_mode(series: pd.Series):
    m = series.mode()
    return m.iloc[0] if not m.empty else np.nan

print("Safe mode on populated series:", get_safe_mode(df_population['Income']))
print("Safe mode on empty series:    ", get_safe_mode(empty_series))

# Unhandled attempt will trigger KeyError:
try:
    bad_call = empty_series.mode()[0]
    print("Mode extraction result:", bad_call)
except Exception as e: # Try KeyError specifically if you want to catch that
    print("Caught unhandled extraction bug:", type(e), e)


Safe mode on populated series: 29515
Safe mode on empty series:     nan
Caught unhandled extraction bug: <class 'KeyError'> 0


---

### Concept 25: Impact of Distributional Skewness on Mean vs. Median vs. Mode

**Question:**
Illustrate the relationship between Mean, Median, and Mode in right-skewed, left-skewed, and symmetric distributions.

**Answer:**
- **Symmetric Distribution (Normal):**
  $$\text{Mean} \approx \text{Median} \approx \text{Mode}$$
- **Right-Skewed (Positive Skew):**
  Long right tail pulls the mean upward:
  $$\text{Mode} < \text{Median} < \text{Mean}$$
- **Left-Skewed (Negative Skew):**
  Long left tail pulls the mean downward:
  $$\text{Mean} < \text{Median} < \text{Mode}$$


In [26]:
# Generating Skewed Distributions
np.random.seed(42)
right_skewed = np.random.exponential(scale=2.0, size=10000)

mean_val = np.mean(right_skewed)
median_val = np.median(right_skewed)

print(f"Right-Skewed Distribution:")
print(f"Median: {median_val:.4f} < Mean: {mean_val:.4f} (Mean pulled right by tail)")


Right-Skewed Distribution:
Median: 1.3566 < Mean: 1.9550 (Mean pulled right by tail)


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import skewnorm
from scipy.optimize import minimize_scalar
from ipywidgets import interact, FloatSlider

def plot_distribution(alpha):
    x = np.linspace(-5, 5, 500)
    y = skewnorm.pdf(x, a=alpha)

    # Calculate mean, median and mode
    mean = skewnorm.mean(a=alpha)
    median = skewnorm.median(a=alpha)

    result = minimize_scalar(
        lambda x: -skewnorm.pdf(x, a=alpha),
        bounds=(-5, 5),
        method="bounded"
    )
    mode = result.x

    # Plot distribution
    plt.figure(figsize=(6, 4))
    plt.plot(x, y, color="skyblue", linewidth=2, label="Density")
    plt.fill_between(x, y, alpha=0.15, color="skyblue")

    # Mean, Median and Mode lines
    plt.axvline(mean, color="hotpink", linestyle="-", linewidth=2,
                label=f"Mean = {mean:.2f}")
    plt.axvline(median, color="gold", linestyle="--", linewidth=2,
                label=f"Median = {median:.2f}")
    plt.axvline(mode, color="limegreen", linestyle=":", linewidth=2.5,
                label=f"Mode = {mode:.2f}")

    plt.title(f"Skew-Normal Distribution (α = {alpha:.1f})")
    plt.xlabel("X")
    plt.ylabel("Density")
    plt.xlim(-5, 5)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

interact(
    plot_distribution,
    alpha=FloatSlider(
        value=0.3,
        min=-10,
        max=10,
        step=0.1,
        description="Skew (α):",
        continuous_update=False
    )
);

interactive(children=(FloatSlider(value=0.3, continuous_update=False, description='Skew (α):', max=10.0, min=-…

---

### Concept 26: Full End-to-End Benchmark Simulation

**Question:**
Using the multi-store retail dataset scenario from the curriculum, demonstrate how Random, Stratified, Cluster, and Systematic sampling estimators compare against the true population mean.

**Answer:**
We execute the end-to-end benchmark on `df_population` (1,000 customers across North, South, East, West store locations) comparing each sample's estimate of average income against the actual population mean $\mu$.


In [28]:
# Multi-Store Retail Benchmark Simulation
np.random.seed(0)
df_population_bench = pd.DataFrame({
    'CustomerID': range(1, 1001),
    'StoreLocation': np.random.choice(['North', 'South', 'East', 'West'], 1000),
    'Satisfaction': np.random.randint(1, 6, 1000),
    'Income': np.random.randint(20000, 80000, 1000)
})

# True Population Parameter
actual_mean_income = df_population_bench['Income'].mean()

# 1. Random Sampling (n=100)
random_sample = df_population_bench.sample(n=100, random_state=1)
random_mean = random_sample['Income'].mean()

# 2. Stratified Sampling (25 per StoreLocation)
stratified_sample = df_population_bench.groupby('StoreLocation', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 25), random_state=1)
)
stratified_mean = stratified_sample['Income'].mean()

# 3. Cluster Sampling (2 random store locations chosen, all customers sampled)
clusters = df_population_bench['StoreLocation'].unique()
selected_clusters = np.random.choice(clusters, size=2, replace=False)
cluster_sample = df_population_bench[df_population_bench['StoreLocation'].isin(selected_clusters)]
cluster_mean = cluster_sample['Income'].mean()

# 4. Systematic Sampling (Every 10th record)
systematic_sample = df_population_bench.iloc[::10, :]
systematic_mean = systematic_sample['Income'].mean()

# Compilation of Results
results_df = pd.DataFrame({
    'Sampling Method': ['Actual Population', 'Random Sampling (n=100)', 'Stratified Sampling (n=100)', 
                        'Cluster Sampling (2 stores)', 'Systematic Sampling (k=10)'],
    'Sample Size': [len(df_population_bench), len(random_sample), len(stratified_sample), 
                    len(cluster_sample), len(systematic_sample)],
    'Estimated Mean Income': [actual_mean_income, random_mean, stratified_mean, cluster_mean, systematic_mean],
    'Absolute Error': [0.0, abs(actual_mean_income - random_mean), abs(actual_mean_income - stratified_mean),
                      abs(actual_mean_income - cluster_mean), abs(actual_mean_income - systematic_mean)]
})

results_df['Error %'] = (results_df['Absolute Error'] / actual_mean_income) * 100
display(results_df)


,Sampling Method,Sample Size,Estimated Mean Income,Absolute Error,Error %
0,Actual Population,1000,49529.357000,0.000000,0.000000
1,Random Sampling (n=100),100,51425.230000,1895.873000,3.827776
2,Stratified Sampling (n=100),100,51507.210000,1977.853000,3.993294
3,Cluster Sampling (2 stores),506,48718.565217,810.791783,1.636992
4,Systematic Sampling (k=10),100,53630.110000,4100.753000,8.279439


---

## Section 5: Practical Coding Exercises

### Exercise 1 (Beginner): Mode and Central Tendency Reporter
**Task:** Write a Python function `compute_summary_stats(series: pd.Series) -> dict` that calculates the mean, median, safe primary mode, count, and standard deviation, gracefully handling missing values and empty series.


In [29]:
def compute_summary_stats(s: pd.Series) -> dict:
    if s.empty or s.dropna().empty:
        return {'mean': np.nan, 'median': np.nan, 'mode': np.nan, 'count': 0, 'std': np.nan}
    
    clean_s = s.dropna()
    mode_vals = clean_s.mode()
    safe_mode = mode_vals.iloc[0] if not mode_vals.empty else np.nan
    
    return {
        'mean': float(clean_s.mean()),
        'median': float(clean_s.median()),
        'mode': safe_mode,
        'count': int(clean_s.count()),
        'std': float(clean_s.std(ddof=1))
    }

# Validation
test_s = pd.Series([100, 200, 200, 300, 400, np.nan])
print("Summary Statistics:", compute_summary_stats(test_s))


Summary Statistics: {'mean': 240.0, 'median': 200.0, 'mode': np.float64(200.0), 'count': 5, 'std': 114.0175425099138}


### Exercise 2 (Intermediate): Stratified Train/Val/Test Splitter from Scratch
**Task:** Build a stratified splitter `stratified_split(df, group_col, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42)` that partitions a dataframe while strictly preserving category distributions.


In [30]:
def stratified_split(df: pd.DataFrame, group_col: str, 
                    train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    assert np.isclose(train_ratio + val_ratio + test_ratio, 1.0)
    
    train_dfs, val_dfs, test_dfs = [], [], []
    
    for _, group in df.groupby(group_col):
        shuffled = group.sample(frac=1.0, random_state=seed)
        n = len(shuffled)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        
        train_dfs.append(shuffled.iloc[:n_train])
        val_dfs.append(shuffled.iloc[n_train:n_train + n_val])
        test_dfs.append(shuffled.iloc[n_train + n_val:])
        
    return pd.concat(train_dfs), pd.concat(val_dfs), pd.concat(test_dfs)

train_df, val_df, test_df = stratified_split(df_population_bench, 'StoreLocation')

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")


Train size: 698, Val size: 148, Test size: 154


### Exercise 3 (Senior / Lead Architect Challenge): Production Reservoir Sampler for Streaming Logs
**Task:** Implement Algorithm R (Reservoir Sampling) to stream arbitrary incoming records of unknown total size $N$, maintaining a uniform random sample of size $k$ in $O(k)$ memory.


In [31]:
class ReservoirSampler:
    """
    Algorithm R for uniform streaming sampling.
    Guarantees every item in a stream of unknown size N has equal probability k/N of inclusion.
    """
    def __init__(self, k: int, seed: int = 42):
        self.k = k
        self.reservoir = []
        self.total_seen = 0
        self.rng = np.random.default_rng(seed)

    def feed(self, item):
        self.total_seen += 1
        if len(self.reservoir) < self.k:
            self.reservoir.append(item)
        else:
            # Generate random integer j in [0, total_seen - 1]
            j = self.rng.integers(0, self.total_seen)
            if j < self.k:
                self.reservoir[j] = item

    def get_sample(self):
        return self.reservoir

# Stream 100,000 simulated sensor records
sampler = ReservoirSampler(k=5)
for event_id in range(1, 100001):
    sampler.feed({'event_id': event_id, 'sensor_val': event_id * 1.5})

print("Streaming Reservoir Sample of 5 elements from 100k stream:")
for record in sampler.get_sample():
    print(record)


Streaming Reservoir Sample of 5 elements from 100k stream:
{'event_id': 49788, 'sensor_val': 74682.0}
{'event_id': 70835, 'sensor_val': 106252.5}
{'event_id': 53491, 'sensor_val': 80236.5}
{'event_id': 24987, 'sensor_val': 37480.5}
{'event_id': 38115, 'sensor_val': 57172.5}


---

## Section 6: Quick Reference Guide

### 1. Sampling Techniques – Comparison Matrix

| Method        | Python Implementation                                                | Best When...                                                         | Critical Limitation                                   |
| ------------- | -------------------------------------------------------------------- | -------------------------------------------------------------------- | ----------------------------------------------------- |
| Simple Random | `df.sample(n=k, replace=False)`                                      | Homogeneous population; complete sampling frame available.           | May under-represent minority subgroups by chance.     |
| Stratified    | `df.groupby('C', group_keys=False).apply(lambda x: x.sample(n=n_c))` | Population has distinct subgroups (strata) that need representation. | Requires prior knowledge of subgroup labels.          |
| Cluster       | `df[df['C'].isin(np.random.choice(df['C'].unique(), size=m))]`       | Population is geographically or logistically divided into clusters.  | High variance when intra-cluster correlation is high. |
| Systematic    | `df.iloc[start::k, :]`                                               | Items arrive continuously on a production line or data stream.       | Vulnerable to hidden periodic patterns.               |

### 2. Measures of Central Tendency – Comparison Matrix

| Metric | Pandas Method | Mathematical Definition             | Breakdown Point | Time Complexity                                        | Sensitivity                                |
| ------ | ------------- | ----------------------------------- | --------------- | ------------------------------------------------------ | ------------------------------------------ |
| Mean   | `s.mean()`    | $\frac{1}{n} \sum_{i=1}^{n} x_i$ | 0%              | $\mathcal{O}(N)$                                             | Highly sensitive to extreme outliers.      |
| Median | `s.median()`  | Middle order statistic              | 50%             | $\mathcal{O}(N \log N)$ sorting; selection can be $\mathcal{O}(N)$. | Robust to the magnitude of extreme values. |
| Mode   | `s.mode()[0]` | Most frequently occurring value     | Variable        | $\mathcal{O}(N)$ average with hash-based counting.           | Suitable for nominal and categorical data. |

Note: Markdown tables do not support automatic column-width adjustment. Text wrapping depends on the Markdown renderer and available screen width.

### 3. Complexity & Profiling Reference
- **Pandas `.mean()` / `.std()`:** $O(N)$ time, $O(1)$ space.
- **Pandas `.median()`:** $O(N)$ average time via introselect, $O(N)$ space.
- **Pandas `.mode()`:** $O(N)$ time using C-hash table, $O(K)$ space for $K$ unique items.


---

## Section 7: Final Interview Takeaways & Architecture Summary

1. **Estimator Precision vs. Cost**: Always default to Stratified Sampling if strata labels are known and sub-population variance is high. Use Cluster Sampling only when collecting the full frame is prohibitively expensive.
2. **Never blindly use `.mode()[0]`**: Empty Series or all-NaN Series will throw `IndexError/KeyError`. Wrap in defensive extraction.
3. **Beware Default Arguments**: `pandas.DataFrame.sample()` defaults to `replace=False`, whereas `numpy.random.choice()` defaults to `replace=True`.
4. **Outlier Strategy**: When designing automated metric monitoring pipelines, track **both** Mean and Median. A divergence between Mean and Median is the most reliable first-principles indicator of skewness or upstream data corruption.
